# Importação das bibliotecas necessárias

In [1]:
import sys
import os
sys.path.append('/home/jovyan/work')

from utils.AlinharETL import AlinharETL

# Crie uma instância da classe AlinharETL

In [2]:
# Crie uma instância da classe
bucket = "silver"
datamart = "SGT"
extrator_bronze = AlinharETL(bucket,datamart)

2024-10-09 14:37:56,493 - INFO - Iniciando Sessão Spark.


# Leitura da tabela

In [3]:
df_atendimentos_sgt = extrator_bronze.criar_view_temporaria('bronze/SGT/AtendimentoSgt','atendimentossgt')

2024-10-09 14:38:00,589 - INFO - Aguarde... Criando View (bronze/SGT/AtendimentoSgt)
2024-10-09 14:38:11,121 - INFO - View criada com sucesso


In [4]:
sql_query = """
    SELECT 
        CRProdutoRegional as cr_produto_regional,
        ano,
        codigoDNProdutoCategoria as nr_dn_produto_categoria,
        codigoDNProdutoLinha as nr_dn_produto_linha,
        codigoDNProdutoNacional as nr_dn_produto_nacional,
        codigoOBAUnidadeOperacional as nr_oab_unidade_operacional,
        cpfCnpjCliente as nr_cpf_cnpj_cliente,
        descricaoProdutoCategoria as dsc_produto_categoria,
        descricaoProdutoLinha as dsc_produto_linha,
        descricaoProdutoNacional as dsc_produto_nacional,
        descricaoProdutoRegional as dsc_produto_regional,
        descricaoStatusAtendimento as dsc_status_atendimento,
        descricaoUnidadeOperacional as dsc_unidade_operacional,
        dia,
        id,
        idAtendimento as id_atendimento,
        CASE 
            WHEN idPorteCliente LIKE '%@nil%' THEN regexp_extract(idPorteCliente, '@nil":"([^"]+)', 1)
            ELSE idPorteCliente 
        END AS id_porte_cliente,
        isCompartilhado as dsc_compartilhamento,
        isEmRede as dsc_em_rede,
        isEscopoIndefinido as dsc_escopo_indefinido,
        isValorHora as dsc_valor_hora,
        mes,
        nomeCliente as nm_cliente,
        CASE 
            WHEN numeroAtendimento LIKE '%@nil%' THEN regexp_extract(numeroAtendimento, '@nil":"([^"]+)', 1)
            ELSE numeroAtendimento 
        END AS nr_atendimento,
        CASE 
            WHEN numeroDeFuncionarios LIKE '%@nil%' THEN 0
            ELSE numeroDeFuncionarios 
        END AS nr_de_funcionario,
        numeroDeProducaoEstimada as nr_de_producao_estimada,
        numeroDeReceitaEstimada as nr_de_receitas_estimada,
        numeroDeRelatorioEstimado as nr_de_relatorio_estimado,
        CASE 
            WHEN porteCliente LIKE '%@nil%' THEN regexp_extract(porteCliente, '@nil":"([^"]+)', 1)
            ELSE porteCliente 
        END AS dsc_porte_cliente,
        tituloAtendimento as dsc_titulo_atendimento,
        vlrEconomico as vlr_economico,
        vlrFinanceiro as vlr_financeiro
    FROM 
        atendimentossgt

"""

In [5]:
df_atendimentos_sgt = extrator_bronze.carregar_dados_delta(sql_query)

2024-10-09 14:38:15,317 - INFO - Aguarde... Executando Query (Delta)
2024-10-09 14:38:15,319 - INFO - 
    SELECT 
        CRProdutoRegional as cr_produto_regional,
        ano,
        codigoDNProdutoCategoria as nr_dn_produto_categoria,
        codigoDNProdutoLinha as nr_dn_produto_linha,
        codigoDNProdutoNacional as nr_dn_produto_nacional,
        codigoOBAUnidadeOperacional as nr_oab_unidade_operacional,
        cpfCnpjCliente as nr_cpf_cnpj_cliente,
        descricaoProdutoCategoria as dsc_produto_categoria,
        descricaoProdutoLinha as dsc_produto_linha,
        descricaoProdutoNacional as dsc_produto_nacional,
        descricaoProdutoRegional as dsc_produto_regional,
        descricaoStatusAtendimento as dsc_status_atendimento,
        descricaoUnidadeOperacional as dsc_unidade_operacional,
        dia,
        id,
        idAtendimento as id_atendimento,
        CASE 
            WHEN idPorteCliente LIKE '%@nil%' THEN regexp_extract(idPorteCliente, '@nil":"([^"]+)', 1

# Gravação no datalake

In [6]:
extrator_bronze.caminho_tabela_delta = 's3a://silver/SGT/AtendimentoSgt'

In [7]:
extrator_bronze.salvar_delta(df_atendimentos_sgt, 'overwrite')

2024-09-25 17:05:29,388 - INFO - Aguarde... Persistindo dados (overwrite)
2024-09-25 17:05:40,322 - INFO - Dados persistidos com sucesso
2024-09-25 17:05:40,323 - INFO - s3a://silver/SGT/AtendimentoSgt
2024-09-25 17:05:40,324 - INFO - Aguarde... Realizando optimize
2024-09-25 17:05:42,748 - INFO - Optimize executado com sucesso.
2024-09-25 17:05:42,748 - INFO - Aguarde... Realizando vacuum
2024-09-25 17:06:09,619 - INFO - Vacuum executado com sucesso.


# Encerra sessão spark

In [8]:
extrator_bronze.parar_sessao()

2024-09-25 17:06:10,237 - INFO - Sessão Spark finalizada.
